# 🚨 RedFlagIQ — Exploratory Analysis Notebook
## Apple Inc. (AAPL) | Financial Statement Red Flag Detection
---
**Project:** RedFlagIQ — Financial Statement Red Flag Detector  
**Company:** Apple Inc. (AAPL)  
**Analysis Period:** FY2021 – FY2024  
**Authors:** Shriya Shetty (Finance & Tech) | Shreshti Shukla (Finance)  

---
### What This Notebook Does
This notebook walks through the **complete forensic financial analysis** of Apple Inc. step by step:

1. Load and inspect real financial data  
2. Explore key financial trends  
3. Calculate the **Beneish M-Score** (earnings manipulation detector)  
4. Calculate the **Altman Z-Score** (bankruptcy risk predictor)  
5. Run **custom red flag checks** (analyst heuristics)  
6. Produce a final risk verdict  

> **Note:** This notebook is the exploratory layer. The production pipeline (`main.py`) automates this across 50+ companies.


## 📦 Step 0 — Setup & Imports

In [ ]:
import sys
import os
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")


sys.path.insert(0, os.path.abspath(".."))

from models.beneish_mscore import calculate_mscore
from models.altman_zscore  import calculate_zscore
from models.ratio_analysis import run_red_flags

print("✅ All imports successful")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")


---
## 📥 Step 1 — Load Apple Financial Data

We load 4 years of Apple financials (FY2021–FY2024).  
All figures are in **USD Millions**.  
Fiscal year ends in **September**.

> In production, `data_fetch.py` pulls this automatically from yfinance.  
> Here we use the pre-verified dataset from our research phase.


In [ ]:


raw_data = {
    
    "revenue":        [391035, 383285, 394328, 365817],
    "cogs":           [210352, 214137, 223546, 212981],
    "gross_profit":   [180683, 169148, 170782, 152836],
    "sga":            [26097,  24932,  25094,  21973],
    "depreciation":   [11445,  11519,  11104,  11284],
    "ebit":           [123216, 114301, 119437, 108949],
    "net_income":     [93736,  96995,  99803,  94680],

    
    "total_assets":   [364980, 352583, 352755, 351002],
    "current_assets": [152987, 143566, 135405, 134836],
    "current_liab":   [176392, 145308, 153982, 125481],
    "working_cap":    [-23405, -1742,  -18577,  9355],
    "accounts_rec":   [33410,  29508,  28184,  26278],
    "long_term_debt": [85750,  95281,  98959, 109106],
    "total_liab":     [308030, 290437, 302083, 287912],
    "retained_earn":  [-19154, -214,   -3068,   5562],
    "intangibles":    [0,      0,      0,       0],

   
    "operating_cf":   [118254, 110543, 122151, 104038],
    "capex":          [9447,   10959,  10708,  11085],
    "free_cf":        [108807, 99584,  111443,  92953],

   
    "market_cap":     [3500000, 3500000, 3500000, 3500000],
}


fiscal_years = pd.to_datetime([
    "2024-09-28",   # FY2024
    "2023-09-30",   # FY2023
    "2022-09-24",   # FY2022
    "2021-09-25",   # FY2021
])

financials = pd.DataFrame(raw_data, index=fiscal_years)
financials.index.name = "fiscal_year_end"

print("✅ Data loaded successfully")
print(f"   Shape: {financials.shape} ({financials.shape[0]} years × {financials.shape[1]} metrics)")
print(f"   Years: {[str(y.date()) for y in financials.index]}")


### 🔍 Preview the Raw Data

In [ ]:
# Display key line items in a readable format
display_cols = ["revenue", "gross_profit", "ebit", "net_income",
                "total_assets", "operating_cf", "free_cf"]

preview = financials[display_cols].copy()
preview.index = ["FY2024", "FY2023", "FY2022", "FY2021"]
preview.index.name = "Year"

# Format as $M with commas
preview_fmt = preview.applymap(lambda x: f"${x:,.0f}M")
print("Apple Inc. (AAPL) — Key Financials (USD Millions)")
print("=" * 75)
print(preview_fmt.to_string())


In [ ]:
# Company metadata — used by scoring models
info = {
    "ticker":        "AAPL",
    "company_name":  "Apple Inc.",
    "sector":        "Technology",
    "industry":      "Consumer Electronics",
    "market_cap":    3_500_000_000_000,   # $3.5 Trillion (live snapshot)
    "currency":      "USD",
}

print(f"Company  : {info['company_name']} ({info['ticker']})")
print(f"Sector   : {info['sector']}  |  Industry: {info['industry']}")
print(f"Market Cap: ${info['market_cap']/1e12:.2f} Trillion")
print(f"Currency : {info['currency']}")


---
## 📊 Step 2 — Exploratory Data Analysis

Before running the models, let's understand Apple's financial trends.  
This is the **qualitative layer** — building intuition before the numbers.


In [ ]:

years   = ["FY2021", "FY2022", "FY2023", "FY2024"]
rev     = financials["revenue"].values[::-1]          
gp      = financials["gross_profit"].values[::-1]
ni      = financials["net_income"].values[::-1]
ocf     = financials["operating_cf"].values[::-1]

print("APPLE INC. — KEY FINANCIAL TRENDS")
print("=" * 65)
print(f"{'Metric':<22} {'FY2021':>10} {'FY2022':>10} {'FY2023':>10} {'FY2024':>10}")
print("-" * 65)

for label, vals in [
    ("Revenue ($M)",       rev),
    ("Gross Profit ($M)",  gp),
    ("Net Income ($M)",    ni),
    ("Operating CF ($M)",  ocf),
]:
    row = f"  {label:<20}"
    for v in vals:
        row += f" {v:>10,.0f}"
    print(row)

print("-" * 65)

print()
print("YEAR-OVER-YEAR GROWTH RATES")
print("=" * 55)
print(f"{'Metric':<22} {'FY22 vs FY21':>12} {'FY23 vs FY22':>12} {'FY24 vs FY23':>12}")
print("-" * 55)

metrics = {
    "Revenue":       rev,
    "Gross Profit":  gp,
    "Net Income":    ni,
    "Operating CF":  ocf,
}

for label, vals in metrics.items():
    growths = []
    for i in range(1, len(vals)):
        g = (vals[i] - vals[i-1]) / abs(vals[i-1])
        growths.append(f"{g:>+11.1%}")
    print(f"  {label:<20}  {'  '.join(growths)}")


### 📐 Margin Analysis
Margins tell us about the **quality** of Apple's earnings — not just the size.


In [ ]:
years_chron = ["FY2021", "FY2022", "FY2023", "FY2024"]
rev_c  = financials["revenue"].values[::-1]
gp_c   = financials["gross_profit"].values[::-1]
ebit_c = financials["ebit"].values[::-1]
ni_c   = financials["net_income"].values[::-1]
ocf_c  = financials["operating_cf"].values[::-1]

gross_margins   = [gp / r  for gp, r  in zip(gp_c,   rev_c)]
ebit_margins    = [e  / r  for e,  r  in zip(ebit_c,  rev_c)]
net_margins     = [n  / r  for n,  r  in zip(ni_c,   rev_c)]
ocf_margins     = [o  / r  for o,  r  in zip(ocf_c,  rev_c)]

print("APPLE INC. — MARGIN TRENDS")
print("=" * 55)
print(f"{'Margin':<22} {'FY2021':>8} {'FY2022':>8} {'FY2023':>8} {'FY2024':>8}")
print("-" * 55)

for label, margins in [
    ("Gross Margin",     gross_margins),
    ("EBIT Margin",      ebit_margins),
    ("Net Margin",       net_margins),
    ("OCF Margin",       ocf_margins),
]:
    row = f"  {label:<20}"
    for m in margins:
        row += f"  {m:>6.1%}"
    print(row)

print("-" * 55)
print()
print("OBSERVATION:")
print("  Gross margin improved from 41.8% (FY21) to 46.2% (FY24) — strong pricing power")
print("  OCF margin is consistently above net margin — earnings are cash-backed")
print("  This is a POSITIVE signal for earnings quality (low manipulation risk)")


### 🏦 Balance Sheet Health Check

In [ ]:
ta_c  = financials["total_assets"].values[::-1]
tl_c  = financials["total_liab"].values[::-1]
ltd_c = financials["long_term_debt"].values[::-1]
re_c  = financials["retained_earn"].values[::-1]
wc_c  = financials["working_cap"].values[::-1]

equity = [a - l for a, l in zip(ta_c, tl_c)]
de     = [d / max(e, 1) for d, e in zip(ltd_c, equity)]

print("APPLE INC. — BALANCE SHEET HEALTH")
print("=" * 65)
print(f"{'Metric':<28} {'FY2021':>8} {'FY2022':>8} {'FY2023':>8} {'FY2024':>8}")
print("-" * 65)

rows = [
    ("Total Assets ($M)",      ta_c,  ",.0f"),
    ("Total Liabilities ($M)", tl_c,  ",.0f"),
    ("Shareholders Equity ($M)",equity,",.0f"),
    ("Long-Term Debt ($M)",    ltd_c, ",.0f"),
    ("Working Capital ($M)",   wc_c,  ",.0f"),
    ("Debt/Equity (x)",        de,    ".2f"),
]

for label, vals, fmt in rows:
    row = f"  {label:<26}"
    for v in vals:
        if ",.0f" in fmt:
            row += f"  {v:>8,.0f}"
        else:
            row += f"  {v:>8.2f}"
    print(row)

print("-" * 65)
print()
print("KEY OBSERVATIONS:")
print("  Negative retained earnings = Apple's aggressive $700B+ share buyback program")
print("  NOT a sign of losses — net income has been positive every year")
print("  Negative working capital is common for large-cap tech firms running lean")
print("  Long-term debt DECLINING: $109B (FY21) -> $86B (FY24) — deleveraging")


---
## 🔴 Step 3 — Beneish M-Score (Earnings Manipulation)

**What is it?**  
The Beneish M-Score was developed by Professor Messod Beneish in 1999.  
It uses 8 financial ratios to estimate the probability that a company has manipulated its earnings.

**Interpretation:**
| M-Score | Signal |
|---------|--------|
| > -1.78 | 🔴 Likely Manipulator — investigate |
| -2.22 to -1.78 | 🟡 Grey Zone — monitor |
| < -2.22 | 🟢 Unlikely Manipulator |

**How it works:**  
Each ratio captures a different manipulation tactic — inflating revenue, slowing depreciation, growing soft assets, or hiding debt. The weighted combination gives a single score.


In [ ]:
# ── Run Beneish M-Score ──────────────────────────────────────────────────────
print("Running Beneish M-Score on Apple (AAPL)...")
print("Comparing: FY2023 (Prior Year) vs FY2024 (Current Year)")
print()

mscore_result = calculate_mscore(financials, info)


### 🔍 Deep Dive — What Each Ratio Is Telling Us

In [ ]:
# Explain each ratio result in plain English
ratio_explanations = {
    "DSRI": {
        "full_name": "Days Sales Receivable Index",
        "what_it_checks": "Are receivables growing faster than sales?",
        "apple_story": (
            "Apple's DSRI = 1.11 (above threshold of 1.031). "
            "Receivables grew 13.2% while revenue grew only 2%. "
            "This COULD signal fake revenue — but for Apple, the more likely explanation "
            "is the rapid expansion of the Services business, which has different "
            "payment cycles than hardware. Worth monitoring but not alarming."
        )
    },
    "GMI": {
        "full_name": "Gross Margin Index",
        "what_it_checks": "Is gross margin shrinking? Pressure -> manipulation.",
        "apple_story": (
            "Apple's GMI = 0.955 (below threshold of 1.014). CLEAN. "
            "Gross margin actually IMPROVED from 44.1% to 46.2%. "
            "No margin pressure = no motivation to manipulate from this angle."
        )
    },
    "AQI": {
        "full_name": "Asset Quality Index",
        "what_it_checks": "Are non-productive (soft) assets growing?",
        "apple_story": (
            "Apple's AQI = 1.000 (below threshold of 1.040). CLEAN. "
            "Apple has essentially zero goodwill/intangibles on its balance sheet. "
            "Its assets are real — cash, PP&E, receivables. No soft asset inflation."
        )
    },
    "SGI": {
        "full_name": "Sales Growth Index",
        "what_it_checks": "Is growth so high it creates pressure to sustain it?",
        "apple_story": (
            "Apple's SGI = 1.020 (below threshold of 1.134). CLEAN. "
            "Revenue grew only 2% YoY — modest, not pressure-zone growth. "
            "No artificial urgency to inflate numbers."
        )
    },
    "DEPI": {
        "full_name": "Depreciation Index",
        "what_it_checks": "Is the company slowing depreciation to inflate profits?",
        "apple_story": (
            "Apple's DEPI = 1.020 (marginally above threshold of 1.001). MILD FLAG. "
            "Depreciation rate slowed very slightly. "
            "Given the scale of Apple's asset base, this is marginal and "
            "likely reflects normal asset lifecycle changes, not manipulation."
        )
    },
    "SGAI": {
        "full_name": "SGA Expense Index",
        "what_it_checks": "Are overheads growing faster than revenue?",
        "apple_story": (
            "Apple's SGAI = 1.026 (below threshold of 1.054). CLEAN. "
            "SG&A grew from 6.5% to 6.7% of revenue — very slightly. "
            "No alarming overhead bloat."
        )
    },
    "TATA": {
        "full_name": "Total Accruals to Total Assets",
        "what_it_checks": "Are earnings backed by real cash (vs paper profits)?",
        "apple_story": (
            "Apple's TATA = -0.067 (well below threshold of 0.018). VERY CLEAN. "
            "Operating CF ($118B) EXCEEDS net income ($94B). "
            "This is the gold standard — Apple earns MORE cash than it reports as profit. "
            "Extremely low manipulation risk on this metric."
        )
    },
    "LVGI": {
        "full_name": "Leverage Index",
        "what_it_checks": "Is debt rising relative to assets? More debt = more pressure.",
        "apple_story": (
            "Apple's LVGI = 1.053 (below threshold of 1.111). CLEAN. "
            "Leverage increased slightly but Apple's debt is actually declining YoY. "
            "The slight increase is driven by current liabilities, not long-term debt."
        )
    },
}

print("BENEISH M-SCORE — RATIO INTERPRETATION")
print("=" * 70)

for ratio, info_r in ratio_explanations.items():
    val   = mscore_result["ratios"][ratio]
    flag  = ratio in mscore_result["flagged_ratios"]
    icon  = "🔴 FLAG" if flag else "✅  OK "
    val_s = f"{val:.4f}" if val == val else "N/A"

    print(f"\n{icon}  {ratio} — {info_r['full_name']}")
    print(f"       Value: {val_s}   |   Checks: {info_r['what_it_checks']}")
    print(f"       Apple: {info_r['apple_story']}")

print()
print("=" * 70)
print(f"  FINAL M-SCORE : {mscore_result['m_score']:.4f}")
print(f"  VERDICT       : {mscore_result['verdict']}")
print(f"  FLAGGED RATIOS: {', '.join(mscore_result['flagged_ratios'])}")
print("=" * 70)


### 📝 Analyst Note — Why Apple's M-Score May Be Misleading

Apple's M-Score comes out higher than expected for a company of its calibre. Here's why:

**The M-Score was built in 1999.** It was designed for traditional manufacturing companies — not capital-light, services-heavy tech giants.

For Apple specifically:
- **Negative retained earnings** is not a red flag — it's a direct result of returning $700B+ to shareholders through buybacks
- **Negative working capital** is intentional and common for Apple's business model (suppliers finance inventory)
- **Services business** creates naturally higher receivables relative to product revenue

**Bottom line:** Always apply analyst judgement on top of model outputs. Models are tools, not verdicts.


---
## 📊 Step 4 — Altman Z-Score (Bankruptcy Risk)

**What is it?**  
Developed by Edward Altman in 1968, the Z-Score predicts financial distress using 5 financial ratios.  
It answers: **"Could this company go bankrupt in the next 2 years?"**

**Interpretation:**
| Z-Score | Zone | Signal |
|---------|------|--------|
| > 2.99 | 🟢 Safe Zone | Low probability of distress |
| 1.81 – 2.99 | 🟡 Grey Zone | Uncertain |
| < 1.81 | 🔴 Distress Zone | High bankruptcy risk |


In [ ]:
# ── Run Altman Z-Score ───────────────────────────────────────────────────────
print("Running Altman Z-Score on Apple (AAPL)...")
print("Using FY2024 data (most recent fiscal year)")
print()

zscore_result = calculate_zscore(financials, info)


### 🔍 Deep Dive — What Each Factor Is Telling Us

In [ ]:
factor_explanations = {
    "X1": {
        "name":   "Working Capital / Total Assets",
        "weight": 1.2,
        "story": (
            "X1 = -0.064. NEGATIVE — Apple has more short-term liabilities than assets. "
            "For most companies this is a red flag. For Apple it's intentional: "
            "they use supplier credit aggressively (accounts payable is massive) "
            "and maintain minimal idle cash in current assets. "
            "This is a feature of Apple's business model, not a weakness."
        )
    },
    "X2": {
        "name":   "Retained Earnings / Total Assets",
        "weight": 1.4,
        "story": (
            "X2 = -0.052. NEGATIVE — But this is entirely explained by buybacks. "
            "Apple has returned so much cash to shareholders that retained earnings "
            "went negative. Net income has been strongly positive every year. "
            "An analyst must NOT interpret this as historical losses."
        )
    },
    "X3": {
        "name":   "EBIT / Total Assets",
        "weight": 3.3,
        "story": (
            "X3 = 0.338. STRONG. Apple generates 33 cents of operating profit "
            "per dollar of assets. This is exceptional efficiency for a company "
            "at this scale. Note: weight of 3.3 makes this the most impactful factor."
        )
    },
    "X4": {
        "name":   "Market Cap / Total Liabilities",
        "weight": 0.6,
        "story": (
            "X4 = 11.36. EXCEPTIONAL. Apple's $3.5T market cap is 11 times "
            "its total liabilities. Even if Apple's assets went to zero, "
            "the market is pricing in massive future cash flows. "
            "This factor single-handedly pushes Apple deep into the safe zone."
        )
    },
    "X5": {
        "name":   "Revenue / Total Assets",
        "weight": 1.0,
        "story": (
            "X5 = 1.071. SOLID. Apple generates more than $1 of revenue "
            "per dollar of assets. Efficient capital utilisation — "
            "characteristic of a capital-light business model."
        )
    },
}

print("ALTMAN Z-SCORE — FACTOR INTERPRETATION")
print("=" * 70)

total_check = 0
for factor, details in factor_explanations.items():
    val      = zscore_result["factors"][factor]
    weighted = zscore_result["weighted"][factor]
    total_check += weighted

    print(f"\n  {factor}  (weight: {details['weight']}×)")
    print(f"       {details['name']}")
    print(f"       Raw value: {val:.4f}  |  Weighted contribution: {weighted:.4f}")
    print(f"       {details['story']}")

print()
print("=" * 70)
print(f"  FINAL Z-SCORE : {zscore_result['z_score']:.4f}")
print(f"  ZONE          : {zscore_result['zone']}")
print(f"  MEANING       : {zscore_result['meaning']}")
print("=" * 70)
print()
print("CONCLUSION: Apple is in the SAFE ZONE with a Z-Score of ~8.85.")
print("Safe zone threshold is 2.99. Apple is nearly 3x above it.")
print("Zero financial distress risk based on this model.")


---
## 🔍 Step 5 — Custom Red Flag Analysis

These are **analyst heuristics** — practical checks that experienced finance analysts run  
beyond the formal models. They add a qualitative judgement layer.

These checks are defined by **Shreshti's Excel framework** and automated here.


In [ ]:
# ── Run Custom Red Flags ─────────────────────────────────────────────────────
print("Running Custom Red Flag checks on Apple (AAPL)...")
print("Comparing: FY2023 (Prior Year) vs FY2024 (Current Year)")
print()

redflags_result = run_red_flags(financials, info)


In [ ]:
# ── Detailed interpretation of each flag ────────────────────────────────────
flag_analysis = [
    {
        "flag":   "Revenue growing but OCF flat",
        "result": "PASS",
        "detail": (
            "Revenue grew 2.0%, OCF grew 7.0%. "
            "OCF is actually growing FASTER than revenue — "
            "a sign of improving cash conversion. Very healthy."
        )
    },
    {
        "flag":   "AR growing 2x faster than revenue",
        "result": "FLAG",
        "detail": (
            "AR grew 13.2% vs revenue growth of 2.0%. "
            "AR growing 6.5x faster than revenue is notable. "
            "Most likely driven by Services segment expansion "
            "(App Store, iCloud, Apple Pay — longer settlement cycles). "
            "Monitor this over the next 2 years. If it persists, dig deeper."
        )
    },
    {
        "flag":   "Gross margin declining 2 consecutive years",
        "result": "PASS",
        "detail": (
            "Gross margin improved from 44.1% to 46.2%. "
            "Trend is positive — Services mix shift is driving "
            "higher-margin revenue. Opposite of a red flag."
        )
    },
    {
        "flag":   "D/E spike > 40% in one year",
        "result": "PASS",
        "detail": (
            "D/E ratio went from 1.53x to 1.51x — slight DECREASE. "
            "Apple is actively deleveraging. Long-term debt declined "
            "from $95B to $86B YoY. No leverage concern whatsoever."
        )
    },
    {
        "flag":   "Net income positive, FCF negative",
        "result": "PASS",
        "detail": (
            "Net Income: $93.7B | FCF: $108.8B. "
            "FCF is actually HIGHER than net income — "
            "the strongest possible signal of earnings quality. "
            "Apple's profits are fully backed (and then some) by real cash."
        )
    },
]

print("CUSTOM RED FLAG — DETAILED INTERPRETATION")
print("=" * 70)
for i, f in enumerate(flag_analysis, 1):
    icon = "🔴 FLAG" if f["result"] == "FLAG" else "✅ PASS"
    print(f"\n  [{i}] {icon} — {f['flag']}")
    print(f"       {f['detail']}")

print()
print("=" * 70)
print(f"  Total Flags Triggered : {redflags_result['triggered_count']} / {redflags_result['total_flags']}")
print(f"  Severity              : {redflags_result['severity']}")
print("=" * 70)


---
## 📋 Step 6 — Final Risk Assessment

Combining all three models into a single verdict.


In [ ]:
# ── Aggregate risk across all three models ───────────────────────────────────
risks = [
    mscore_result["risk"],
    zscore_result["risk"],
    redflags_result["severity"],
]

if "HIGH" in risks:
    overall_risk = "HIGH"
elif risks.count("MEDIUM") >= 2:
    overall_risk = "MEDIUM"
elif "MEDIUM" in risks:
    overall_risk = "MEDIUM"
else:
    overall_risk = "LOW"

risk_icons = {"HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🟢"}

print("╔══════════════════════════════════════════════════════════════╗")
print(f"║   REDFLAGIQ — FINAL RISK REPORT                              ║")
print(f"║   {info['company_name']:<58}║")
print(f"║   {info['ticker']:<58}║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║                                                              ║")
print(f"║   Beneish M-Score  : {mscore_result['m_score']:>8.4f}  |  Risk: {mscore_result['risk']:<16}  ║")
print(f"║   Altman Z-Score   : {zscore_result['z_score']:>8.4f}  |  Zone: {zscore_result['risk']:<16}  ║")
print(f"║   Custom Flags     : {redflags_result['triggered_count']:>2} / {redflags_result['total_flags']}       |  Severity: {redflags_result['severity']:<10}  ║")
print(f"║                                                              ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║                                                              ║")
print(f"║   OVERALL RISK RATING:  {risk_icons[overall_risk]}  {overall_risk:<36}║")
print(f"║                                                              ║")
print("╚══════════════════════════════════════════════════════════════╝")


### 📝 Key Analyst Findings

In [ ]:
findings = [
    ("LOW RISK overall",
     "All three models point to a fundamentally healthy company. "
     "No coordinated red flags across models."),

    ("M-Score grey zone — but context matters",
     "M-Score of ~2.89 is technically above the manipulation threshold, "
     "but driven by Apple's unique business characteristics, not manipulation. "
     "The 1999 model was not designed for capital-light services businesses."),

    ("AR growth is the one metric to watch",
     "Receivables growing 6.5x faster than revenue is the only genuine concern. "
     "Most likely Services-driven, but should be tracked for 2-3 more years."),

    ("Earnings quality is exceptional",
     "TATA of -0.067 and FCF > Net Income confirms Apple's earnings are "
     "fully cash-backed. This is the single most important earnings quality metric."),

    ("Zero financial distress risk",
     "Z-Score of 8.85 vs a distress threshold of 1.81. Apple's market cap "
     "alone ($3.5T) is 11x its total liabilities. Bankruptcy risk is negligible."),

    ("Negative retained earnings and working capital — not red flags",
     "Both are accounting artefacts of Apple's $700B+ buyback program and "
     "its supplier financing model. An analyst must contextualise, not react blindly."),
]

print("KEY FINDINGS — Apple Inc. (AAPL) | FY2024 Analysis")
print("=" * 70)
for i, (title, detail) in enumerate(findings, 1):
    print(f"\n  {i}. {title}")
    print(f"     {detail}")
print()
print("=" * 70)
print()
print("RECOMMENDATION: LOW risk. No investigative action warranted.")
print("Monitor AR/Revenue ratio in FY2025 results as a single data point to watch.")


---
## 💾 Step 7 — Export Results for PowerBI

This CSV is what Shreshti's PowerBI dashboard reads directly.  
Run `main.py` in production to generate this for multiple companies automatically.


In [ ]:
import os

# Build results row — same format as main.py pipeline output
result_row = {
    "ticker":                 info["ticker"],
    "company_name":           info["company_name"],
    "sector":                 info["sector"],
    "analysis_year":          mscore_result["curr_year"],
    "prior_year":             mscore_result["prev_year"],
    "m_score":                mscore_result["m_score"],
    "m_score_verdict":        mscore_result["verdict"],
    "m_score_risk":           mscore_result["risk"],
    "m_flagged_ratios":       ", ".join(mscore_result["flagged_ratios"]),
    "z_score":                zscore_result["z_score"],
    "z_score_zone":           zscore_result["zone"],
    "z_score_risk":           zscore_result["risk"],
    "custom_flags_triggered": redflags_result["triggered_count"],
    "custom_flags_total":     redflags_result["total_flags"],
    "custom_flags_severity":  redflags_result["severity"],
    "flag_1_revenue_ocf":     "FLAG" if redflags_result["flags"][0]["triggered"] else "OK",
    "flag_2_ar_revenue":      "FLAG" if redflags_result["flags"][1]["triggered"] else "OK",
    "flag_3_gross_margin":    "FLAG" if redflags_result["flags"][2]["triggered"] else "OK",
    "flag_4_debt_equity":     "FLAG" if redflags_result["flags"][3]["triggered"] else "OK",
    "flag_5_ni_fcf":          "FLAG" if redflags_result["flags"][4]["triggered"] else "OK",
    "overall_risk":           overall_risk,
}

df_result = pd.DataFrame([result_row])

# Save to outputs/results.csv
os.makedirs("../outputs", exist_ok=True)
csv_path = "../outputs/results.csv"
df_result.to_csv(csv_path, index=False)

print(f"✅ Results exported to {csv_path}")
print(f"   Shreshti's PowerBI dashboard reads this file directly.")
print()
print("Preview:")
print(df_result[["ticker","m_score","z_score","custom_flags_triggered","overall_risk"]].to_string(index=False))


---
## 🚀 What's Next

This notebook covers the full exploratory analysis for AAPL.  

**To run the production pipeline:**
```bash
# Single company
python main.py --ticker AAPL

# Multiple companies
python main.py --batch tickers.csv
```

**To extend this analysis:**
- Add more companies to `tickers.csv` and run batch mode
- Shreshti connects `outputs/results.csv` to PowerBI for the dashboard
- Compare Apple against peers (MSFT, GOOGL, AMZN) for benchmarking

---
*Built by Shriya Shetty & Shreshti Shukla | RedFlagIQ Portfolio Project*
